### Reading Delta Tables

In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
df = spark.read.table("databricks_etl_catalog.bronze.regions")

In [ ]:
df.display()

In [ ]:
df = df.drop("_rescued_data")

In [ ]:
df.write.format("delta")\
    .mode("overwrite")\
    .save(f"abfss://silver@{storage_account}.dfs.core.windows.net/regions")

### Reading all the data from all three Layers & Tables 

In [ ]:
df = spark.read.format("delta").load(f"abfss://silver@{storage_account}.dfs.core.windows.net/regions")
df.display()

In [ ]:
df = spark.read.format("delta").load(f"abfss://silver@{storage_account}.dfs.core.windows.net/orders")
df.display()

In [ ]:
df = spark.read.format("delta").load(f"abfss://silver@{storage_account}.dfs.core.windows.net/customers")
df.display()

In [ ]:
df = spark.read.format("delta").load(f"abfss://silver@{storage_account}.dfs.core.windows.net/products")
df.display()

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS databricks_etl_catalog.silver.regions_silver
USING DELTA
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/regions'

In [ ]:
%sql
Select * from databricks_etl_catalog.silver.regions_silver